In [1]:
%%writefile worker.py
import numpy as np
import astropy.constants as const
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
from lenstronomy.LensModel.lens_model import LensModel
import warnings

warnings.filterwarnings("ignore")

cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
all_bands = [
    # 'g', 'r', 'i', 'z', 'y',
    'F062', 'F106', 'F129', 'F158', 'F184'
]


def extract_lens_properties(args):
    lens_id, lens = args
    
    deflector = lens.deflector
    source = lens.source(index=0)
    z_D = lens.deflector_redshift
    z_S = lens.source_redshift_list[0]
    theta_E = lens.einstein_radius[0]
    num_images = lens.image_number[0]

    xs, ys = lens.source(0).extended_source_position
    radial_dist_S = np.sqrt(xs**2 + ys**2)
    size_S = source.angular_size 

    mags = {}
    snrs = {}
    contrasts = {}
    
    fov_arcsec = np.max([theta_E * 4, 1])
    
    for b in all_bands:
        # Map to the correct observatory for SNR
        obs = 'LSST' if b in ['g', 'r', 'i', 'z', 'y'] else 'Roman'

        mags[f'mag_S_{b}'] = source.extended_source_magnitude(b)
        mags[f'mag_S_{b}_lensed'] = lens.extended_source_magnitude(band=b, lensed=True)[0]
        mags[f'mag_D_{b}'] = deflector.magnitude(b)
        
        # Calculate SNR
        snrs[f'snr_{b}'] = lens.snr(
            band=b,
            fov_arcsec=fov_arcsec,
            observatory=obs, 
            snr_per_pixel_threshold=1,
            exposure_time=642,  # seconds for Roman HLWAS Medium
        )
        
        # Calculate Contrast Ratio
        cr_raw = lens.contrast_ratio(band=b, source_index=0)
        cr_padded = np.array(list(cr_raw) + [np.nan] * (4 - len(cr_raw)))
        contrasts[f'contrast_ratio_{b}'] = cr_padded

    es_magnification = lens.extended_source_magnification[0]

    sigma_v_D = deflector.velocity_dispersion()
    stellar_mass_D = deflector.stellar_mass
    e1_mass_D, e2_mass_D = deflector.mass_ellipticity
    e_mass_D = np.sqrt(e1_mass_D**2 + e2_mass_D**2)
    gamma_pl = deflector.halo_properties.get('gamma_pl', 2.0)
    size_D = deflector.angular_size_light

    lenstronomy_kwargs = lens.lenstronomy_kwargs()
    lens_model_lenstronomy = LensModel(lens_model_list=lenstronomy_kwargs[0]["lens_model_list"])
    lenstronomy_kwargs_lens = lenstronomy_kwargs[1]["kwargs_lens"]
    
    deflector_center = deflector.deflector_center
    grid = np.linspace(-size_D, size_D, 500)
    grid_x, grid_y = np.meshgrid(grid + deflector_center[0], grid + deflector_center[1])
    
    kappa_map = lens_model_lenstronomy.kappa(grid_x, grid_y, kwargs=lenstronomy_kwargs_lens)
    mask = np.sqrt((grid_x - deflector_center[0])**2 + (grid_y - deflector_center[1])**2) < size_D / 2
    kappa_within_half_light_radii = np.nanmean(kappa_map[mask])

    D_s = cosmo.angular_diameter_distance(z_S)
    D_d = cosmo.angular_diameter_distance(z_D)
    D_ds = cosmo.angular_diameter_distance_z1z2(z_D, z_S)
    
    sigma_crit = ((const.c**2 / (4 * np.pi * const.G)) * (D_s / (D_d * D_ds))).to(u.Msun / u.pc**2).value
    surface_density = sigma_crit * kappa_within_half_light_radii

    surface_brightness_map = deflector.surface_brightness(grid_x, grid_y, band="g")
    mask_sb = np.sqrt((grid_x - deflector_center[0])**2 + (grid_y - deflector_center[1])**2) < size_D
    mean_surface_brightness = np.nanmean(surface_brightness_map[mask_sb])

    R_e_kpc_val = (cosmo.kpc_proper_per_arcmin(z_D) * ((size_D * u.arcsec).to(u.arcmin))).to(u.kpc).value

    return {
        "lens_id": lens_id, 
        "z_D": z_D, "z_S": z_S, "theta_E": theta_E, "num_images": num_images,
        "radial_dist_S": radial_dist_S, "size_S": size_S,
        **mags,
        **snrs,
        "es_magnification": es_magnification,
        "R_e_arcsec": size_D, "surf_bri_mag/arcsec2": mean_surface_brightness,
        "sigma_v_D": sigma_v_D, "stellar_mass_D": stellar_mass_D,
        "e1_mass_D": e1_mass_D, "e2_mass_D": e2_mass_D, "e_mass_D": e_mass_D,
        "gamma_pl": gamma_pl, 
        "R_e_kpc": R_e_kpc_val, "Sigma_half_Msun/pc2": surface_density,
        **contrasts
    }

Overwriting worker.py


In [2]:
import os
import copy
import numpy as np
import pandas as pd
import multiprocessing
import speclite
import speclite.filters
from tqdm import tqdm
from astropy.table import Table
from astropy.cosmology import FlatLambdaCDM
from astropy.units import Quantity

import slsim.Pipelines as pipelines
import slsim.Sources as sources
import slsim.Deflectors as deflectors
from slsim.Lenses.lens_pop import LensPop
from slsim.Pipelines import roman_speclite
from warnings import filterwarnings

filterwarnings("ignore", category=UserWarning, append=True)
filterwarnings("ignore", category=RuntimeWarning, append=True)

# Import the worker function
from worker import extract_lens_properties

# --- 1. Setup skypy_config & Roman Filters ---
skypy_config = "lsst-like_triple_SF.yml"

roman_speclite.configure_roman_filters()
roman_filters = roman_speclite.filter_names()
speclite.filters.load_filters(*roman_filters)

# --- 2. Configuration ---
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
sky_area = Quantity(value=10, unit="deg2")

# --- 3. Generate Catalogs In-Memory ---
galaxy_simulation_pipeline = pipelines.SkyPyPipeline(
    skypy_config=skypy_config, sky_area=sky_area, filters=None, cosmo=cosmo
)

red_galaxy_catalog = galaxy_simulation_pipeline.red_galaxies
blue_galaxy_catalog = galaxy_simulation_pipeline.blue_galaxies

kwargs_deflector_cut = {"band": "g", "band_max": 28, "z_min": 0.01, "z_max": 3}
kwargs_source_cut = {"band": "g", "band_max": 28, "z_min": 0.1, "z_max": 6.0}

lens_galaxies = deflectors.EllipticalLensGalaxies(
    galaxy_list=red_galaxy_catalog,
    kwargs_cut=kwargs_deflector_cut,
    kwargs_mass2light=None,
    cosmo=cosmo,
    sky_area=sky_area,
    gamma_pl={"mean": 2.078, "std_dev": 0.16},
)

source_galaxies = sources.Galaxies(
    galaxy_list=blue_galaxy_catalog,
    kwargs_cut=kwargs_source_cut,
    cosmo=cosmo,
    sky_area=sky_area,
    catalog_type="skypy",
    extended_source_type="single_sersic",
)

gg_lens_pop = LensPop(
    deflector_population=lens_galaxies,
    source_population=source_galaxies,
    cosmo=cosmo,
    sky_area=sky_area,
)

In [ ]:
# --- 4. Extract & Compile ---
scale_factor = 1 
all_results = []
num_processes = multiprocessing.cpu_count() - 3
multiprocessing.set_start_method("spawn", force=True)

for i in tqdm(range(scale_factor), desc="Drawing and Processing Lenses"):
    loaded_lenses = gg_lens_pop.draw_population(kwargs_lens_cuts={}, speed_factor=1)
    
    lens_ids = [f"lens_{i}_{j}" for j in range(len(loaded_lenses))]
    lens_args = list(zip(lens_ids, loaded_lenses))
    
    with multiprocessing.Pool(processes=num_processes) as pool:
        results_list = list(tqdm(
            pool.imap(extract_lens_properties, lens_args),
            total=len(loaded_lenses),
            desc="Calculating lens properties",
            leave=False
        ))
        
    all_results.extend([res for res in results_list if res is not None])

# Master Table
if all_results:
    final_catalog = Table.from_pandas(pd.DataFrame(all_results))
    print(f"\nSuccessfully generated {len(final_catalog)} lenses.")
else:
    final_catalog = None
    print("\nNo results were generated.")

Drawing and Processing Lenses: 100%|██████████| 1/1 [01:52<00:00, 112.06s/it]


Successfully generated 2876 lenses in memory.


In [ ]:
# --- 5. Cutting Logic ---
def satisfies_Collett_2015_cuts_table(lens_catalog_table, seeing=0.5, snr_threshold={'i': 20}, 
                                      redshift_limit_source=None, contrast_ratio_threshold_i_band=None, 
                                      return_boolean_array=False):
    """
    Checks whether lenses in a given lens catalog table satisfy the Collett 2015 cuts.
    """
    conditions = np.ones(len(lens_catalog_table), dtype=bool)

    # Criteria 1: Multiple imaging
    conditions &= lens_catalog_table['radial_dist_S'] < lens_catalog_table['theta_E']
    
    # Criteria 2: Image Resolution
    conditions &= lens_catalog_table['theta_E'] > np.sqrt(lens_catalog_table['size_S']**2 + (seeing/2)**2)
    
    # Criteria 3: Tangential Arc Resolution
    conditions &= lens_catalog_table['es_magnification'] * lens_catalog_table['size_S'] > seeing
    
    # Criteria 4: Magnification
    conditions &= lens_catalog_table['es_magnification'] > 3

    # Criteria 5: SNR > 20
    if snr_threshold is not None:
        for band, threshold in snr_threshold.items():
            if f'snr_{band}' in lens_catalog_table.colnames:
                snr_band = lens_catalog_table[f'snr_{band}'].copy()
                snr_band[snr_band == None] = 0  # Treat None SNR values as 0 for the purpose of this cut
                conditions &= snr_band > threshold
            
    # Criteria 6: Contrast Ratio (not in original Collett 2015 cuts, but often used in lens selection)
    # at least two images with contrast ratio > threshold in the i-band
    if contrast_ratio_threshold_i_band is not None and 'contrast_ratio_i' in lens_catalog_table.colnames:
        conditions_6 = np.ones(len(lens_catalog_table), dtype=bool)
        contrast_ratio_i = lens_catalog_table['contrast_ratio_i'] # this is in mags I_source_light/I_lens_light [mag/arcsec^2]
        for i in range(len(lens_catalog_table)):
            cr_i_lens = 10**(-contrast_ratio_i[i] / 2.5) # convert to flux ratio 
            cr_i_lens = cr_i_lens[~np.isnan(cr_i_lens)]  # remove nan values (for images that don't exist)
            if np.sum(cr_i_lens > contrast_ratio_threshold_i_band) < 2:
                conditions_6[i] = False
        conditions &= conditions_6

    # Criteria 7: Redshift cuts for Source if needed
    if redshift_limit_source is not None:
        conditions &= lens_catalog_table['z_S'] < redshift_limit_source
        
    if return_boolean_array:
        return conditions
    return lens_catalog_table[conditions]

In [18]:
filtered_catalog = satisfies_Collett_2015_cuts_table(
    final_catalog, 
    seeing=0.2, 
    snr_threshold={'F106': 20}, 
    redshift_limit_source=None, 
    contrast_ratio_threshold_i_band=2, 
    return_boolean_array=False
)
filtered_catalog

lens_id,z_D,z_S,theta_E,num_images,radial_dist_S,size_S,mag_S_F062,mag_S_F062_lensed,mag_D_F062,mag_S_F106,mag_S_F106_lensed,mag_D_F106,mag_S_F129,mag_S_F129_lensed,mag_D_F129,mag_S_F158,mag_S_F158_lensed,mag_D_F158,mag_S_F184,mag_S_F184_lensed,mag_D_F184,snr_F062,snr_F106,snr_F129,snr_F158,snr_F184,es_magnification,R_e_arcsec,surf_bri_mag/arcsec2,sigma_v_D,stellar_mass_D,e1_mass_D,e2_mass_D,e_mass_D,gamma_pl,R_e_kpc,Sigma_half_Msun/pc2,contrast_ratio_F062,contrast_ratio_F106,contrast_ratio_F129,contrast_ratio_F158,contrast_ratio_F184
str11,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,object,object,object,object,object
lens_0_2,0.6151633758714046,1.885131649140829,0.745262014517735,2,0.0915419351281081,0.07669886610614654,24.671949544894574,21.65698822740127,21.223525374072963,23.588740331770595,20.573779014277292,19.452185986718284,22.930673701690466,19.915712384197164,19.049686761492886,22.579247868015777,19.564286550522475,18.645293911290516,22.319301865728995,19.304340548235693,18.415833695400508,155.0150791100802,260.5853290714373,371.05181227350266,442.59861134648736,398.6613086862975,16.06884002081944,0.6806370536681159,24.35493679144786,216.94705654932955,319177907714.50037,-0.01478076209180107,0.08241209859274089,0.08372708595474984,2.0895881922642627,4.602774125941603,5066.764949185301,[-3.19768176 -2.55162729 nan nan],[-2.50955158 -1.86349711 nan nan],[-2.76511899 -2.11906452 nan nan],[-2.71215197 -2.0660975 nan nan],[-2.74263776 -2.09658329 nan nan]
lens_0_4,0.3017097510647348,3.6527008707084647,0.9855729558390651,2,0.4706959931859024,0.07523064776284483,25.059570762876156,23.273897621112972,19.204670678781778,24.901786866121135,23.11611372435795,17.920350991269597,24.872467145514364,23.08679400375118,17.519003092346164,24.722302025824625,22.93662888406144,17.256128658520733,24.38166963250167,22.595996490738486,17.11362389920883,52.68081475247059,48.46803737464456,44.708951760568915,47.43223602678127,45.574548445710114,5.17927840686388,0.7006934101469277,22.25177876802944,185.80053497956678,203034454743.5227,-0.11336158469232202,-0.01307492795729677,0.11411311329134351,2.160324171923236,3.1333879328063605,6230.357744603889,[-2.08738784 0.4253239 nan nan],[-0.96085205 1.55185969 nan nan],[-0.58882388 1.92388787 nan nan],[-0.47611456 2.03659719 nan nan],[-0.6742422 1.83846955 nan nan]
lens_0_6,0.7919131006835405,2.975073576584274,0.26311773387681153,3,0.04242885320964644,0.01869282244724363,27.40343704341531,24.74083894391484,24.971040137800472,26.96110566791524,24.298507568414774,22.897618966075626,26.663552163231763,24.000954063731296,22.52071390613243,26.084650040246984,23.422051940746517,22.198804750880612,25.720265596388238,23.05766749688777,21.945109109630458,29.990271788167355,31.55519392920926,30.11009802291973,43.253399748608906,40.581700767661204,11.615535589234934,0.11625291596155447,24.179794824546335,124.09507211514384,28170605699.59937,0.07507787774014966,-0.16103374036040427,0.17767541546434318,1.7293202640646808,0.8698282884134834,5570.640757890867,[-6.35782048 -3.68595496 1.71708849 nan],[-4.72673069 -2.05486517 3.34817828 nan],[-4.64737913 -1.97551361 3.42752984 nan],[-4.9043721 -2.23250658 3.17053687 nan],[-5.0150609 -2.34319538 3.05984807 nan]
lens_0_24,1.2738342139908292,2.144296968825582,0.26465602047713965,2,0.21068399102084326,0.053186271779581545,25.496619469151746,24.03098487899677,25.305775014733566,24.910554335615807,23.44491974546083,22.672626042830583,24.351758755669096,22.88612416551412,22.12505546338115,24.01720139289673,22.551566802741753,21.737889627070754,23.84317384727847,22.377539257123495,21.48849713778605,41.52364036070936,53.259569234516086,64.86586347848413,76.67898926594194,64.82374267931213,3.857037275774109,0.3898508685050313,26.

In [19]:
# Roman Medium HLWAS sky area is 2415 deg^2, so we can scale our results accordingly to estimate the number of lenses that would be found in the actual survey.
print(f"\nEstimated number of lenses in Roman Medium HLWAS (2415 deg^2): {len(filtered_catalog) * (2415 / sky_area.value):.0f}")


Estimated number of lenses in Roman Medium HLWAS (2415 deg^2): 71726
